# EdTech Learning Outcome Analytics

## Day 2: Data Cleaning and Preprocessing

### Objective

The objective of this notebook is to prepare the OULAD datasets for further analysis.

The cleaning process includes:

- Checking data quality
- Handling missing values based on business meaning
- Checking duplicate records
- Validating data types
- Checking invalid or inconsistent values
- Preparing cleaned datasets without modifying the original raw data

The cleaned datasets will be saved in:

`../data/processed/`

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
raw_data_path = "../data/raw"
processed_data_path = "../data/processed"

In [3]:
os.makedirs(processed_data_path, exist_ok=True)

print("Raw data path :", raw_data_path)
print("Processed data path :", processed_data_path) 

Raw data path : ../data/raw
Processed data path : ../data/processed


In [4]:
courses = pd.read_csv("../data/raw/courses.csv")
assessments = pd.read_csv("../data/raw/assessments.csv")
student_assessment = pd.read_csv("../data/raw/studentAssessment.csv")
student_info = pd.read_csv("../data/raw/studentInfo.csv")
student_registration = pd.read_csv("../data/raw/studentRegistration.csv")
student_vle = pd.read_csv("../data/raw/studentVle.csv")
vle = pd.read_csv("../data/raw/vle.csv")

In [5]:
datasets = {
    "courses" : courses,
    "assessments" : assessments,
    "student_assessment": student_assessment,
    "student_info" : student_info,
    "student_registration" : student_registration,
    "student_vle" : student_vle,
    "vle" : vle
}

In [6]:
quality_report = []

for name, df in datasets.items():
    quality_report.append({
        "Dataset" : name,
        "Rows" : df.shape[0],
        "Columns" : df.shape[1],
        "Missing Values" : df.isnull().sum().sum(),
        "Duplicate Rows" : df.duplicated().sum()
    })

quality_report = pd.DataFrame(quality_report)
quality_report

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,courses,22,3,0,0
1,assessments,206,6,0,0
2,student_assessment,173912,5,0,0
3,student_info,32593,12,0,0
4,student_registration,32593,5,0,0
5,student_vle,10655280,6,0,787170
6,vle,6364,6,0,0


In [7]:
missing_values = pd.DataFrame()

for name, df in datasets.items():
    missing = df.isnull().sum()

    missing = missing[missing>0]

    if not missing.empty:
        temp = pd.DataFrame({
            "Dataset" : name,
            "Column":missing.index,
            "Missing Values":missing.values,
            "Missing Percentage" : (
                missing.values / len(df) * 100
            ).round(2)
        })
        missing_values = pd.concat(
            [missing_values,temp],
            ignore_index=True
        )
missing_values

""


## 1. Cleaning the Courses Dataset

The courses dataset contains information about course modules and their presentations.

The key objective is to verify:

- Missing values
- Duplicate records
- Valid course duration values
- Uniqueness of course and presentation combinations

In [8]:
courses.info()

<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   code_module                 22 non-null     str  
 1   code_presentation           22 non-null     str  
 2   module_presentation_length  22 non-null     int64
dtypes: int64(1), str(2)
memory usage: 836.0 bytes


In [9]:
courses.isnull().sum()

code_module                   0
code_presentation             0
module_presentation_length    0
dtype: int64

In [10]:
courses.duplicated().sum()

np.int64(0)

In [11]:
courses.describe()

,module_presentation_length
count,22.000000
mean,255.545455
std,13.654677
min,234.000000
25%,241.000000
50%,261.500000
75%,268.000000
max,269.000000


In [12]:
courses["module_presentation_length"].describe()

count     22.000000
mean     255.545455
std       13.654677
min      234.000000
25%      241.000000
50%      261.500000
75%      268.000000
max      269.000000
Name: module_presentation_length, dtype: float64

In [13]:
courses[courses["module_presentation_length"] <= 0]

,code_module,code_presentation,module_presentation_length


In [14]:
courses.duplicated(
    subset=["code_module", "code_presentation"]
).sum()

np.int64(0)

In [15]:
courses[
    courses.duplicated(
        subset=["code_module", "code_presentation"],
        keep=False
    )
]

,code_module,code_presentation,module_presentation_length


## 2. Cleaning the Assessments Dataset

The assessments dataset contains information about course assessments, including assessment type, due date, and weight.

We need to verify:

- Missing values
- Duplicate assessment records
- Invalid weights
- Assessment types
- Missing due dates

In [16]:
assessments.info()

<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    str    
 1   code_presentation  206 non-null    str    
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    str    
 4   date               206 non-null    str    
 5   weight             206 non-null    float64
dtypes: float64(1), int64(1), str(4)
memory usage: 12.5 KB


In [17]:
assessments.isnull().sum()

code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

In [18]:
assessments.duplicated().sum()

np.int64(0)

In [19]:
assessments["assessment_type"].value_counts()

assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

In [20]:
assessments["weight"].describe()

count    206.000000
mean      20.873786
std       30.384224
min        0.000000
25%        0.000000
50%       12.500000
75%       24.250000
max      100.000000
Name: weight, dtype: float64

In [21]:
assessments[
    assessments["weight"] < 0
]

,code_module,code_presentation,id_assessment,assessment_type,date,weight


In [22]:
assessments[
    assessments["date"].isnull()
]

,code_module,code_presentation,id_assessment,assessment_type,date,weight


## 3. Cleaning the Student Assessment Dataset

This dataset records student assessment submissions and scores.

We will check:

- Missing values
- Duplicate records
- Score ranges
- Submission dates
- Valid student and assessment identifiers

In [23]:
student_assessment.info()

<class 'pandas.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   id_assessment   173912 non-null  int64
 1   id_student      173912 non-null  int64
 2   date_submitted  173912 non-null  int64
 3   is_banked       173912 non-null  int64
 4   score           173912 non-null  str  
dtypes: int64(4), str(1)
memory usage: 7.0 MB


In [24]:
student_assessment.isnull().sum()

id_assessment     0
id_student        0
date_submitted    0
is_banked         0
score             0
dtype: int64

In [25]:
student_assessment.duplicated().sum()

np.int64(0)

In [26]:
student_assessment["score"].describe()

count     173912
unique       102
top          100
freq       18813
Name: score, dtype: object

In [27]:
student_assessment["score"] = pd.to_numeric(
    student_assessment["score"],
    errors="coerce"
)

In [28]:
invalid_scores = student_assessment[
    (student_assessment["score"] < 0) |
    (student_assessment["score"] > 100)
]

invalid_scores

,id_assessment,id_student,date_submitted,is_banked,score


In [29]:
student_assessment["date_submitted"].describe()

count    173912.000000
mean        116.032942
std          71.484148
min         -11.000000
25%          51.000000
50%         116.000000
75%         173.000000
max         608.000000
Name: date_submitted, dtype: float64

In [30]:
student_assessment[
    student_assessment["date_submitted"] < 0
]

,id_assessment,id_student,date_submitted,is_banked,score
1641,1758,603861,-1,1,61.0
1685,1758,1970390,-1,1,78.0
1690,1758,2073016,-1,1,60.0
1693,1758,2139353,-1,1,65.0
1708,1758,2358969,-1,1,68.0
...,...,...,...,...,...
172516,37440,624767,-1,1,100.0
172526,37440,631446,-1,1,100.0
172717,37441,624767,-1,1,100.0
172727,37441,631446,-1,1,100.0


In [31]:
student_assessment[
    student_assessment["date_submitted"] < 0
].groupby("is_banked").size()

is_banked
0     148
1    1909
dtype: int64

## 4. Cleaning the Student Information Dataset

The student information dataset contains demographic, academic, and final outcome information.

We will validate:

- Missing values
- Duplicate student-course records
- Categorical values
- Final results
- Numeric ranges

In [32]:
student_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code_module           32593 non-null  str  
 1   code_presentation     32593 non-null  str  
 2   id_student            32593 non-null  int64
 3   gender                32593 non-null  str  
 4   region                32593 non-null  str  
 5   highest_education     32593 non-null  str  
 6   imd_band              32593 non-null  str  
 7   age_band              32593 non-null  str  
 8   num_of_prev_attempts  32593 non-null  int64
 9   studied_credits       32593 non-null  int64
 10  disability            32593 non-null  str  
 11  final_result          32593 non-null  str  
dtypes: int64(3), str(9)
memory usage: 4.8 MB


In [33]:
student_info.isnull().sum()

code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

In [34]:
student_info.duplicated().sum()

np.int64(0)

In [35]:
categorical_columns = student_info.select_dtypes(
    include="object"
).columns

for column in categorical_columns:
    print(f"\n{'=' * 50}")
    print(column)
    print(student_info[column].value_counts(dropna=False))


code_module
code_module
BBB    7909
FFF    7762
DDD    6272
CCC    4434
EEE    2934
GGG    2534
AAA     748
Name: count, dtype: int64

code_presentation
code_presentation
2014J    11260
2013J     8845
2014B     7804
2013B     4684
Name: count, dtype: int64

gender
gender
M    17875
F    14718
Name: count, dtype: int64

region
region
Scotland                3446
East Anglian Region     3340
London Region           3216
South Region            3092
North Western Region    2906
West Midlands Region    2582
South West Region       2436
East Midlands Region    2365
South East Region       2111
Wales                   2086
Yorkshire Region        2006
North Region            1823
Ireland                 1184
Name: count, dtype: int64

highest_education
highest_education
A Level or Equivalent          14045
Lower Than A Level             13158
HE Qualification                4730
No Formal quals                  347
Post Graduate Qualification      313
Name: count, dtype: int64

imd_band
imd

C:\Users\Rushi\AppData\Local\Temp\ipykernel_5412\2903998645.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = student_info.select_dtypes(


In [36]:
student_info["final_result"].value_counts(dropna=False)

final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

In [37]:
student_info.describe()

,id_student,num_of_prev_attempts,studied_credits
count,3.259300e+04,32593.000000,32593.000000
mean,7.066877e+05,0.163225,79.758691
std,5.491673e+05,0.479758,41.071900
min,3.733000e+03,0.000000,30.000000
25%,5.085730e+05,0.000000,60.000000
50%,5.903100e+05,0.000000,60.000000
75%,6.444530e+05,0.000000,120.000000
max,2.716795e+06,6.000000,655.000000


In [38]:
student_info[
    (student_info["num_of_prev_attempts"] < 0) |
    (student_info["studied_credits"] < 0)
]

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result


In [39]:
student_info.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

## 5. Cleaning the Student Registration Dataset

This dataset contains student registration and unregistration dates.

Special attention is required because missing unregistration dates may have business meaning.

In [40]:
student_registration.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   code_module          32593 non-null  str  
 1   code_presentation    32593 non-null  str  
 2   id_student           32593 non-null  int64
 3   date_registration    32593 non-null  str  
 4   date_unregistration  32593 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.6 MB


In [41]:
student_registration.isnull().sum()

code_module            0
code_presentation      0
id_student             0
date_registration      0
date_unregistration    0
dtype: int64

In [42]:
student_registration[
    student_registration["date_unregistration"].isnull()
].head()

,code_module,code_presentation,id_student,date_registration,date_unregistration


In [43]:
student_registration.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()


np.int64(0)

In [44]:
student_registration["date_registration"].describe()   

count     32593
unique      333
top         -22
freq       1034
Name: date_registration, dtype: object

In [45]:
student_registration["date_unregistration"].value_counts(dropna=False).head(20)

date_unregistration
?      22521
12       788
0        419
27       170
-1       147
-2        99
-7        99
10        91
-5        90
-8        87
5         84
-12       80
-4        76
-10       73
9         72
-15       72
11        71
-11       70
51        64
-3        63
Name: count, dtype: int64

In [46]:
student_registration[
    student_registration["date_unregistration"] == "?"
].shape

(22521, 5)

In [47]:
student_registration["date_registration"].value_counts().sort_index().head(20)

date_registration
-1       58
-10     208
-100    208
-101    164
-102    245
-103    117
-104     22
-105     26
-106    185
-107    180
-108    181
-109    225
-11     107
-110     90
-111     16
-112     36
-113    186
-114    146
-115    218
-116    152
Name: count, dtype: int64

In [48]:
student_registration["date_unregistration"].value_counts().sort_index().tail(20)

date_unregistration
82       23
83       17
84        9
85        7
86       31
87       33
88       25
89       36
9        72
90       30
91        2
92        3
93       34
94       41
95       37
96       37
97       50
98        5
99        5
?     22521
Name: count, dtype: int64

## Step 13 Findings and Cleaning Decision

The `student_registration` dataset contains 32,593 records and no actual Pandas null values.

However, `date_unregistration` contains 22,521 records represented by `?`.

In OULAD, the `?` value represents the absence of a recorded unregistration date. These records should not be deleted because the information is meaningful for student retention and withdrawal analysis.

The date fields are stored as strings and contain relative day values. Negative values are valid because OULAD represents dates relative to the start of a course presentation.

### Cleaning decisions

- Convert `date_registration` from string to numeric.
- Convert `date_unregistration` from string to numeric.
- Convert `?` in `date_unregistration` to `NaN`.
- Preserve negative relative dates.
- Do not remove students with missing unregistration dates.

In [49]:
student_registration_clean = student_registration.copy()

In [50]:
student_registration_clean["date_unregistration"] = (
    student_registration_clean["date_unregistration"]
    .replace("?", np.nan)
)

In [51]:
student_registration_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   code_module          32593 non-null  str  
 1   code_presentation    32593 non-null  str  
 2   id_student           32593 non-null  int64
 3   date_registration    32593 non-null  str  
 4   date_unregistration  10072 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.6 MB


In [52]:
student_registration[
    pd.to_numeric(
        student_registration["date_registration"],
        errors="coerce"
    ).isna()
]["date_registration"].value_counts(dropna=False)

date_registration
?    45
Name: count, dtype: int64

In [53]:
student_registration_clean["date_registration"] = (
    student_registration_clean["date_registration"]
    .replace("?", np.nan)
)

student_registration_clean["date_unregistration"] = (
    student_registration_clean["date_unregistration"]
    .replace("?", np.nan)
)

In [54]:
student_registration_clean["date_registration"] = pd.to_numeric(
    student_registration_clean["date_registration"],
    errors="coerce"
)

student_registration_clean["date_unregistration"] = pd.to_numeric(
    student_registration_clean["date_unregistration"],
    errors="coerce"
)

In [55]:
student_registration_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  str    
 1   code_presentation    32593 non-null  str    
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 1.5 MB


In [56]:
student_registration_clean.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

In [57]:
print(
    "Registration '?' values:",
    (student_registration_clean["date_registration"] == "?").sum()
)

print(
    "Unregistration '?' values:",
    (student_registration_clean["date_unregistration"] == "?").sum()
)

Registration '?' values: 0
Unregistration '?' values: 0


In [58]:
invalid_dates = student_registration_clean[
    student_registration_clean["date_unregistration"].notna() &
    (
        student_registration_clean["date_unregistration"]
        < student_registration_clean["date_registration"]
    )
]

invalid_dates.shape

(0, 5)

## 6. Cleaning the Student VLE Dataset

The student VLE dataset records student interactions with the Virtual Learning Environment.

We will check:

- Missing values
- Duplicate interactions
- Click values
- Student activity dates
- Identifier consistency

In [59]:
student_vle.info()

<class 'pandas.DataFrame'>
RangeIndex: 10655280 entries, 0 to 10655279
Data columns (total 6 columns):
 #   Column             Dtype
---  ------             -----
 0   code_module        str  
 1   code_presentation  str  
 2   id_student         int64
 3   id_site            int64
 4   date               int64
 5   sum_click          int64
dtypes: int64(4), str(2)
memory usage: 569.1 MB


In [60]:
student_vle.isnull().sum()

code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

In [61]:
student_vle.duplicated().sum(0)

C:\Users\Rushi\AppData\Local\Temp\ipykernel_5412\1136700816.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  student_vle.duplicated().sum(0)


np.int64(787170)

In [62]:
student_vle["sum_click"].describe()

count    1.065528e+07
mean     3.716946e+00
std      8.849047e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.977000e+03
Name: sum_click, dtype: float64

In [63]:
student_vle[
    student_vle["sum_click"] < 0
]

,code_module,code_presentation,id_student,id_site,date,sum_click


In [64]:
student_vle["date"].describe()

count    1.065528e+07
mean     9.517400e+01
std      7.607130e+01
min     -2.500000e+01
25%      2.500000e+01
50%      8.600000e+01
75%      1.560000e+02
max      2.690000e+02
Name: date, dtype: float64

In [65]:
student_vle[
    student_vle.duplicated(keep=False)
].head(20)

,code_module,code_presentation,id_student,id_site,date,sum_click
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
62,AAA,2013J,45462,546652,-10,1
63,AAA,2013J,45462,546652,-10,1
178,AAA,2013J,77367,546652,-10,4
180,AAA,2013J,77367,546652,-10,4
191,AAA,2013J,94961,546652,-10,2
193,AAA,2013J,94961,546652,-10,2
432,AAA,2013J,248270,546652,-10,4
442,AAA,2013J,248270,546652,-10,4


In [66]:
print("Total rows:", len(student_vle))
print("Unique rows:", student_vle.drop_duplicates().shape[0])
print("Duplicate rows:", student_vle.duplicated().sum())

Total rows: 10655280
Unique rows: 9868110
Duplicate rows: 787170


In [67]:
student_vle.duplicated(
    subset=[
        "code_module",
        "code_presentation",
        "id_student",
        "id_site",
        "date"
    ]
).sum()

np.int64(2195960)

In [68]:
duplicate_groups = (
    student_vle
    .groupby([
        "code_module",
        "code_presentation",
        "id_student",
        "id_site",
        "date"
    ])["sum_click"]
    .nunique()
)

duplicate_groups.value_counts().sort_index()

sum_click
1    7280246
2     997867
3     139730
4      35050
5       5880
6        500
7         36
8         11
Name: count, dtype: int64

In [69]:
different_click_groups = duplicate_groups[
    duplicate_groups > 1
]

different_click_groups.count()

np.int64(1179074)

In [70]:
different_click_keys = duplicate_groups[
    duplicate_groups > 1
].index

In [71]:
example_different_clicks = student_vle.merge(
    different_click_keys.to_frame(index=False),
    on=[
        "code_module",
        "code_presentation",
        "id_student",
        "id_site",
        "date"
    ],
    how="inner"
)

In [72]:
example_different_clicks.head(30)

,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546652,-10,8
4,AAA,2013J,30268,546652,-10,3
5,AAA,2013J,30268,546652,-10,2
6,AAA,2013J,32885,546652,-10,1
7,AAA,2013J,32885,546652,-10,3
8,AAA,2013J,32885,546652,-10,4
9,AAA,2013J,38053,546652,-10,5


In [73]:
exact_duplicate_count = student_vle.duplicated().sum()

student_vle_exact_clean = student_vle.drop_duplicates()

print("Original rows:", len(student_vle))
print("Rows after exact duplicate removal:", len(student_vle_exact_clean))
print("Rows removed:", len(student_vle) - len(student_vle_exact_clean))

Original rows: 10655280
Rows after exact duplicate removal: 9868110
Rows removed: 787170


In [74]:
student_vle_exact_clean

,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1
5,AAA,2013J,28400,546652,-10,8
...,...,...,...,...,...,...
10655274,GGG,2014J,679769,896943,269,2
10655275,GGG,2014J,675811,896943,269,3
10655276,GGG,2014J,675578,896943,269,1
10655277,GGG,2014J,654064,896943,269,3


In [75]:
print("Cleaned VLE rows:", len(student_vle_exact_clean))

Cleaned VLE rows: 9868110


In [76]:
vle_students = set(student_vle_exact_clean["id_student"])
info_students = set(student_info["id_student"])

print("VLE students:", len(vle_students))
print("Student info students:", len(info_students))

print("VLE students not in student_info:",
      len(vle_students - info_students))

print("Student info students not in VLE:",
      len(info_students - vle_students))

VLE students: 26074
Student info students: 28785
VLE students not in student_info: 0
Student info students not in VLE: 2711


In [77]:
print(
    "VLE modules:",
    sorted(student_vle_exact_clean["code_module"].unique())
)

print(
    "Student info modules:",
    sorted(student_info["code_module"].unique())
)

VLE modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
Student info modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']


In [78]:
print(
    "VLE presentations:",
    sorted(student_vle_exact_clean["code_presentation"].unique())
)

print(
    "Student info presentations:",
    sorted(student_info["code_presentation"].unique())
)

VLE presentations: ['2013B', '2013J', '2014B', '2014J']
Student info presentations: ['2013B', '2013J', '2014B', '2014J']


In [79]:
student_vle_exact_clean.duplicated().sum()

np.int64(0)

## Step 14 Findings and Cleaning Decision

The student VLE dataset contains 10,655,280 interaction records.

No missing values or negative click values were identified.

A total of 787,170 exact duplicate records were identified and removed. However, repeated combinations of student, course, presentation, VLE material, and date were not automatically removed because some repeated event keys contained different `sum_click` values.

After exact duplicate removal, 9,868,110 records remained.

All VLE student IDs were successfully matched to the student information dataset. A total of 2,711 students in the student information dataset had no recorded VLE activity; these students were retained because absence of engagement is analytically meaningful for dropout analysis.

Module and presentation identifiers were consistent across the datasets, and the cleaned VLE dataset contains no remaining exact duplicates.

## Step 15. Cleaning the VLE Metadata Dataset

The VLE metadata dataset contains information about the learning materials available in the Virtual Learning Environment.

The `id_site` field connects this dataset to the `student_vle` interaction dataset.

We will validate:

- Dataset structure
- Missing values
- Duplicate VLE resources
- VLE resource identifiers
- Activity types
- Module and presentation consistency
- Identifier relationships with student VLE activity

In [80]:
vle.info()

<class 'pandas.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_site            6364 non-null   int64
 1   code_module        6364 non-null   str  
 2   code_presentation  6364 non-null   str  
 3   activity_type      6364 non-null   str  
 4   week_from          6364 non-null   str  
 5   week_to            6364 non-null   str  
dtypes: int64(1), str(5)
memory usage: 407.2 KB


In [81]:
vle.head()

,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,?,?
1,546712,AAA,2013J,oucontent,?,?
2,546998,AAA,2013J,resource,?,?
3,546888,AAA,2013J,url,?,?
4,547035,AAA,2013J,resource,?,?


In [82]:
vle.isnull().sum()

id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64

In [83]:
vle.duplicated().sum()

np.int64(0)

In [84]:
vle.shape

(6364, 6)

In [85]:
print("Unique id_site values:", vle["id_site"].nunique())
print("Total rows:", len(vle))

Unique id_site values: 6364
Total rows: 6364


In [86]:
vle["id_site"].duplicated().sum()

np.int64(0)

In [87]:
vle.duplicated(
    subset=[
        "code_module",
        "code_presentation",
        "id_site"
    ]
).sum()

np.int64(0)

In [88]:
vle["activity_type"].value_counts(dropna=False)

activity_type
resource          2660
subpage           1055
oucontent          996
url                886
forumng            194
quiz               127
page               102
oucollaborate       82
questionnaire       61
ouwiki              49
dataplus            28
externalquiz        26
homepage            22
glossary            21
ouelluminate        21
dualpane            20
repeatactivity       5
htmlactivity         4
sharedsubpage        3
folder               2
Name: count, dtype: int64

In [89]:
print(
    "VLE modules:",
    sorted(vle["code_module"].unique())
)

print(
    "Student VLE modules:",
    sorted(student_vle_exact_clean["code_module"].unique())
)

VLE modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']


Student VLE modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']


In [90]:
print(
    "VLE presentations:",
    sorted(vle["code_presentation"].unique())
)

print(
    "Student VLE presentations:",
    sorted(
        student_vle_exact_clean["code_presentation"].unique()
    )
)

VLE presentations: ['2013B', '2013J', '2014B', '2014J']
Student VLE presentations: ['2013B', '2013J', '2014B', '2014J']


In [91]:
vle_sites = set(vle["id_site"])
student_vle_sites = set(
    student_vle_exact_clean["id_site"]
)

print("VLE metadata sites:", len(vle_sites))
print("Student VLE sites:", len(student_vle_sites))

print(
    "Student VLE sites missing from VLE metadata:",
    len(student_vle_sites - vle_sites)
)

print(
    "VLE metadata sites not used in student activity:",
    len(vle_sites - student_vle_sites)
)

VLE metadata sites: 6364
Student VLE sites: 6268
Student VLE sites missing from VLE metadata: 0
VLE metadata sites not used in student activity: 96


In [92]:
vle[["week_from", "week_to"]].describe()

,week_from,week_to
count,6364,6364
unique,31,31
top,?,?
freq,5243,5243


In [93]:
print(
    "week_from missing:",
    vle["week_from"].isnull().sum()
)

print(
    "week_to missing:",
    vle["week_to"].isnull().sum()
)

week_from missing: 0
week_to missing: 0


In [94]:
vle[
    vle["week_from"].notna() &
    vle["week_to"].notna() &
    (vle["week_to"] < vle["week_from"])
]

,id_site,code_module,code_presentation,activity_type,week_from,week_to


In [95]:
print("week_from '?' values:",
      (vle["week_from"] == "?").sum())

print("week_to '?' values:",
      (vle["week_to"] == "?").sum())

week_from '?' values: 5243
week_to '?' values: 5243


In [96]:
vle_clean = vle.copy()

In [97]:
vle_clean["week_from"] = pd.to_numeric(
    vle_clean["week_from"].replace("?", np.nan),
    errors="coerce"
)

vle_clean["week_to"] = pd.to_numeric(
    vle_clean["week_to"].replace("?", np.nan),
    errors="coerce"
)

In [98]:
vle_clean = vle.copy()

In [99]:
vle_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_site            6364 non-null   int64
 1   code_module        6364 non-null   str  
 2   code_presentation  6364 non-null   str  
 3   activity_type      6364 non-null   str  
 4   week_from          6364 non-null   str  
 5   week_to            6364 non-null   str  
dtypes: int64(1), str(5)
memory usage: 407.2 KB


In [100]:
vle_clean.isnull().sum()

id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64

In [101]:
invalid_week_ranges = vle_clean[
    vle_clean["week_from"].notna() &
    vle_clean["week_to"].notna() &
    (vle_clean["week_to"] < vle_clean["week_from"])
]

invalid_week_ranges.shape

(0, 6)

In [102]:
vle_clean.duplicated().sum()

np.int64(0)

## Step 16. Cleaning the Assessments Dataset

The assessments dataset contains information about assessments associated with each course presentation.

We will validate:

- Dataset structure
- Missing values
- Duplicate assessment records
- Assessment identifiers
- Assessment types
- Assessment dates
- Assessment weights
- Module and presentation consistency
- Assessment identifier relationships

In [103]:
assessments.info()

<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    str    
 1   code_presentation  206 non-null    str    
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    str    
 4   date               206 non-null    str    
 5   weight             206 non-null    float64
dtypes: float64(1), int64(1), str(4)
memory usage: 12.5 KB


In [104]:
assessments.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10.0
1,AAA,2013J,1753,TMA,54,20.0
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
4,AAA,2013J,1756,TMA,215,30.0


In [105]:
assessments.isnull().sum()

code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

In [106]:
assessments.shape

(206, 6)

In [107]:
assessments.duplicated().sum()

np.int64(0)

In [108]:
print("Total assessment rows:", len(assessments))
print("Unique assessment IDs:", assessments["id_assessment"].nunique())
print(
    "Duplicate assessment IDs:",
    assessments["id_assessment"].duplicated().sum()
)

Total assessment rows: 206
Unique assessment IDs: 206
Duplicate assessment IDs: 0


In [109]:
assessments["assessment_type"].value_counts(dropna=False)

assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

In [110]:
print(
    "Assessment modules:",
    sorted(assessments["code_module"].unique())
)

print(
    "Student info modules:",
    sorted(student_info["code_module"].unique())
)

Assessment modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
Student info modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']


In [111]:
print(
    "Assessment presentations:",
    sorted(assessments["code_presentation"].unique())
)

print(
    "Student info presentations:",
    sorted(student_info["code_presentation"].unique())
)

Assessment presentations: ['2013B', '2013J', '2014B', '2014J']
Student info presentations: ['2013B', '2013J', '2014B', '2014J']


In [112]:
assessments["date"].describe()

count     206
unique     75
top       222
freq       15
Name: date, dtype: object

In [113]:
print("Minimum assessment date:", assessments["date"].min())
print("Maximum assessment date:", assessments["date"].max())

Minimum assessment date: 102
Maximum assessment date: ?


In [114]:
print(
    "Assessment date '?' values:",
    (assessments["date"] == "?").sum()
)

Assessment date '?' values: 11


In [115]:
print(
    sorted(assessments["date"].unique(), key=str)
)

['102', '109', '110', '111', '114', '116', '117', '12', '123', '124', '129', '131', '136', '137', '144', '146', '149', '151', '152', '158', '159', '165', '166', '170', '171', '173', '18', '187', '19', '194', '195', '199', '20', '200', '201', '206', '207', '208', '214', '215', '222', '227', '228', '229', '23', '235', '236', '24', '240', '241', '25', '261', '32', '33', '40', '41', '47', '51', '52', '53', '54', '61', '62', '67', '68', '74', '79', '81', '82', '87', '88', '89', '94', '96', '?']


In [116]:
assessments_clean = assessments.copy()

In [117]:
assessments_clean["date"] = pd.to_numeric(
    assessments_clean["date"].replace("?", np.nan),
    errors="coerce"
)

In [118]:
assessments_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    str    
 1   code_presentation  206 non-null    str    
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    str    
 4   date               195 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), str(3)
memory usage: 12.0 KB


In [119]:
assessments_clean.isnull().sum()

code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

In [120]:
print(
    "Minimum assessment date:",
    assessments_clean["date"].min()
)

print(
    "Maximum assessment date:",
    assessments_clean["date"].max()
)

Minimum assessment date: 12.0
Maximum assessment date: 261.0


In [121]:
assessments_clean[
    assessments_clean["date"] < 0
]

,code_module,code_presentation,id_assessment,assessment_type,date,weight


In [122]:
assessments_clean["date"].describe()

count    195.000000
mean     145.005128
std       76.001119
min       12.000000
25%       71.000000
50%      152.000000
75%      222.000000
max      261.000000
Name: date, dtype: float64

In [123]:
print(
    "Assessment date '?' values:",
    (assessments["date"] == "?").sum()
)

Assessment date '?' values: 11


In [124]:
print(
    sorted(assessments["date"].unique(), key=str)
)

['102', '109', '110', '111', '114', '116', '117', '12', '123', '124', '129', '131', '136', '137', '144', '146', '149', '151', '152', '158', '159', '165', '166', '170', '171', '173', '18', '187', '19', '194', '195', '199', '20', '200', '201', '206', '207', '208', '214', '215', '222', '227', '228', '229', '23', '235', '236', '24', '240', '241', '25', '261', '32', '33', '40', '41', '47', '51', '52', '53', '54', '61', '62', '67', '68', '74', '79', '81', '82', '87', '88', '89', '94', '96', '?']


In [125]:
assessments_clean["weight"].describe()

count    206.000000
mean      20.873786
std       30.384224
min        0.000000
25%        0.000000
50%       12.500000
75%       24.250000
max      100.000000
Name: weight, dtype: float64

In [126]:
print(
    "Weights <= 0:",
    (assessments_clean["weight"] <= 0).sum()
)

print(
    "Weights > 100:",
    (assessments_clean["weight"] > 100).sum()
)

Weights <= 0: 56
Weights > 100: 0


In [127]:
assessment_weights = (
    assessments_clean
    .groupby([
        "code_module",
        "code_presentation"
    ])["weight"]
    .sum()
)

assessment_weights

code_module  code_presentation
AAA          2013J                200.0
             2014J                200.0
BBB          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
CCC          2014B                300.0
             2014J                300.0
DDD          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
EEE          2013J                200.0
             2014B                200.0
             2014J                200.0
FFF          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
GGG          2013J                100.0
             2014B                100.0
             2014J                100.0
Name: weight, dtype: float64

In [128]:
assessments_clean.duplicated(
    subset=[
        "code_module",
        "code_presentation",
        "id_assessment"
    ]
).sum()

np.int64(0)

In [129]:
assessments_clean["weight"].describe()

print(
    "Weights <= 0:",
    (assessments_clean["weight"] <= 0).sum()
)

print(
    "Weights > 100:",
    (assessments_clean["weight"] > 100).sum()
)

Weights <= 0: 56
Weights > 100: 0


In [130]:
assessment_weights = (
    assessments_clean
    .groupby([
        "code_module",
        "code_presentation"
    ])["weight"]
    .sum()
)

assessment_weights

code_module  code_presentation
AAA          2013J                200.0
             2014J                200.0
BBB          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
CCC          2014B                300.0
             2014J                300.0
DDD          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
EEE          2013J                200.0
             2014B                200.0
             2014J                200.0
FFF          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
GGG          2013J                100.0
             2014B                100.0
             2014J                100.0
Name: weight, dtype: float64

In [131]:
assessments_clean.duplicated(
    subset=[
        "code_module",
        "code_presentation",
        "id_assessment"
    ]
).sum()

np.int64(0)

In [132]:
ccc_assessments = assessments_clean[
    assessments_clean["code_module"] == "CCC"
]

ccc_assessments[
    [
        "code_module",
        "code_presentation",
        "id_assessment",
        "assessment_type",
        "date",
        "weight"
    ]
].sort_values(
    [
        "code_presentation",
        "assessment_type",
        "date"
    ]
)

,code_module,code_presentation,id_assessment,assessment_type,date,weight
54,CCC,2014B,24286,CMA,18.0,2.0
55,CCC,2014B,24287,CMA,67.0,7.0
56,CCC,2014B,24288,CMA,137.0,8.0
57,CCC,2014B,24289,CMA,207.0,8.0
62,CCC,2014B,24290,Exam,NaN,100.0
63,CCC,2014B,40087,Exam,NaN,100.0
58,CCC,2014B,24282,TMA,32.0,9.0
59,CCC,2014B,24283,TMA,102.0,22.0
60,CCC,2014B,24284,TMA,151.0,22.0
61,CCC,2014B,24285,TMA,200.0,22.0


In [133]:
ccc_assessments.groupby(
    [
        "code_presentation",
        "assessment_type"
    ]
)["weight"].agg(
    ["count", "sum"]
)

count    sum
code_presentation assessment_type              
2014B             CMA                  4   25.0
                  Exam                 2  200.0
                  TMA                  4   75.0
2014J             CMA                  4   25.0
                  Exam                 2  200.0
                  TMA                  4   75.0

In [134]:
print("Final assessments rows:", len(assessments_clean))
print("Final assessments columns:", assessments_clean.shape[1])

print(
    "Exact duplicates:",
    assessments_clean.duplicated().sum()
)

print(
    "Duplicate assessment keys:",
    assessments_clean.duplicated(
        subset=[
            "code_module",
            "code_presentation",
            "id_assessment"
        ]
    ).sum()
)

print(
    "Missing values:"
)

print(assessments_clean.isnull().sum())

Final assessments rows: 206
Final assessments columns: 6
Exact duplicates: 0
Duplicate assessment keys: 0
Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64


## Step 17. Cleaning the Student Assessment Dataset

The student assessment dataset contains student-level assessment results.

We will validate the dataset structure, missing values, duplicate records, assessment identifiers, scores, and relationships with the assessment metadata.

In [135]:
student_assessment.info()

<class 'pandas.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id_assessment   173912 non-null  int64  
 1   id_student      173912 non-null  int64  
 2   date_submitted  173912 non-null  int64  
 3   is_banked       173912 non-null  int64  
 4   score           173739 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 6.6 MB


In [136]:
student_assessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [137]:
student_assessment.isnull().sum()

id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

In [138]:
student_assessment.shape

(173912, 5)

In [139]:
print("Total student-assessment records:", len(student_assessment))

Total student-assessment records: 173912


In [140]:
student_assessment.duplicated().sum()

np.int64(0)

In [141]:
student_assessment.duplicated(
    subset=[
        "id_assessment",
         "id_student"
    ]
)

0         False
1         False
2         False
3         False
4         False
          ...  
173907    False
173908    False
173909    False
173910    False
173911    False
Length: 173912, dtype: bool

In [142]:
student_assessment["score"].describe()

count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64

In [143]:
print(
    "Scores below 0:",
    (student_assessment["score"] < 0).sum()
)

print(
    "Scores above 100:",
    (student_assessment["score"] > 100).sum()
)

Scores below 0: 0
Scores above 100: 0


In [144]:
student_assessment["is_banked"].value_counts(dropna=False)

is_banked
0    172003
1      1909
Name: count, dtype: int64

In [145]:
print(
    "Invalid is_banked values:",
    (~student_assessment["is_banked"].isin([0, 1])).sum()
)

Invalid is_banked values: 0


In [146]:
student_assessment.duplicated().sum()

np.int64(0)

In [147]:
student_assessment.duplicated(
    subset=[
        "id_assessment",
        "id_student"
    ]
).sum()

np.int64(0)

In [148]:
missing_score_rows = student_assessment[
    student_assessment["score"].isnull()
]

missing_score_rows.head(20)

,id_assessment,id_student,date_submitted,is_banked,score
215,1752,721259,22,0,NaN
937,1754,260355,127,0,NaN
2364,1760,2606802,180,0,NaN
3358,14984,186780,77,0,NaN
3914,14984,531205,26,0,NaN
3943,14984,534151,7,0,NaN
4206,14984,549713,84,0,NaN
4311,14984,554393,55,0,NaN
4753,14985,186780,77,0,NaN
5722,14986,33666,117,0,NaN


In [149]:
print("Missing score records:", len(missing_score_rows))

print(
    "Missing scores - banked status:"
)

print(
    missing_score_rows["is_banked"].value_counts(dropna=False)
)

Missing score records: 173
Missing scores - banked status:
is_banked
0    172
1      1
Name: count, dtype: int64


In [150]:
student_assessment["date_submitted"].describe()

count    173912.000000
mean        116.032942
std          71.484148
min         -11.000000
25%          51.000000
50%         116.000000
75%         173.000000
max         608.000000
Name: date_submitted, dtype: float64

In [151]:
print(
    "Minimum date_submitted:",
    student_assessment["date_submitted"].min()
)

print(
    "Maximum date_submitted:",
    student_assessment["date_submitted"].max()
)

print(
    "Negative date_submitted values:",
    (student_assessment["date_submitted"] < 0).sum()
)

Minimum date_submitted: -11
Maximum date_submitted: 608
Negative date_submitted values: 2057


In [152]:
missing_score_by_assessment = (
    student_assessment[
        student_assessment["score"].isnull()
    ]
    .groupby("id_assessment")
    .size()
    .sort_values(ascending=False)
)

missing_score_by_assessment

id_assessment
25339    7
15013    7
34860    6
25363    5
14984    5
        ..
34887    1
34886    1
34903    1
37415    1
37427    1
Length: 78, dtype: int64

In [153]:
missing_score_by_banked = (
    missing_score_rows["is_banked"]
    .value_counts()
    .rename_axis("is_banked")
    .reset_index(name="missing_scores")
)

missing_score_by_banked["percentage"] = (
    missing_score_by_banked["missing_scores"]
    / len(missing_score_rows)
    * 100
)

missing_score_by_banked

,is_banked,missing_scores,percentage
0,0,172,99.421965
1,1,1,0.578035


In [154]:
assessment_ids = set(assessments_clean["id_assessment"])

student_assessment_ids = set(
    student_assessment["id_assessment"]
)

print(
    "Student assessment IDs:",
    len(student_assessment_ids)
)

print(
    "Assessment metadata IDs:",
    len(assessment_ids)
)

print(
    "Assessment IDs missing from metadata:",
    len(student_assessment_ids - assessment_ids)
)

print(
    "Metadata assessment IDs not used:",
    len(assessment_ids - student_assessment_ids)
)

Student assessment IDs: 188
Assessment metadata IDs: 206
Assessment IDs missing from metadata: 0
Metadata assessment IDs not used: 18


In [155]:
assessment_students = set(
    student_assessment["id_student"]
)

info_students = set(
    student_info["id_student"]
)

print(
    "Students in assessment data:",
    len(assessment_students)
)

print(
    "Students in student info:",
    len(info_students)
)

print(
    "Assessment students missing from student info:",
    len(assessment_students - info_students)
)

print(
    "Student info students without assessments:",
    len(info_students - assessment_students)
)

Students in assessment data: 23369
Students in student info: 28785
Assessment students missing from student info: 0
Student info students without assessments: 5416


In [156]:
missing_score_details = (
    missing_score_rows
    .merge(
        assessments_clean[
            [
                "id_assessment",
                "code_module",
                "code_presentation",
                "assessment_type",
                "date",
                "weight"
            ]
        ],
        on="id_assessment",
        how="left"
    )
)

missing_score_details[
    [
        "id_assessment",
        "code_module",
        "code_presentation",
        "assessment_type",
        "date",
        "weight",
        "is_banked"
    ]
].head(20)

,id_assessment,code_module,code_presentation,assessment_type,date,weight,is_banked
0,1752,AAA,2013J,TMA,19.0,10.0,0
1,1754,AAA,2013J,TMA,117.0,20.0,0
2,1760,AAA,2014J,TMA,117.0,20.0,0
3,14984,BBB,2013B,TMA,19.0,5.0,0
4,14984,BBB,2013B,TMA,19.0,5.0,0
5,14984,BBB,2013B,TMA,19.0,5.0,0
6,14984,BBB,2013B,TMA,19.0,5.0,0
7,14984,BBB,2013B,TMA,19.0,5.0,0
8,14985,BBB,2013B,TMA,47.0,18.0,0
9,14986,BBB,2013B,TMA,89.0,18.0,0


In [157]:
vle_clean = vle.copy()

In [158]:
missing_score_details["assessment_type"].value_counts(
    dropna=False
)

assessment_type
TMA    173
Name: count, dtype: int64

In [159]:
print(
    "Missing scores by assessment type:"
)

print(
    missing_score_details["assessment_type"]
    .value_counts(dropna=False)
)

Missing scores by assessment type:
assessment_type
TMA    173
Name: count, dtype: int64


In [160]:
negative_submission_dates = student_assessment[
    student_assessment["date_submitted"] < 0
]

negative_submission_dates.head(20)

,id_assessment,id_student,date_submitted,is_banked,score
1641,1758,603861,-1,1,61.0
1685,1758,1970390,-1,1,78.0
1690,1758,2073016,-1,1,60.0
1693,1758,2139353,-1,1,65.0
1708,1758,2358969,-1,1,68.0
1735,1758,2576122,-1,1,64.0
1768,1758,65002,-1,1,66.0
1776,1758,94961,-1,1,74.0
1787,1758,129955,-1,1,85.0
1808,1758,159954,-1,1,65.0


In [161]:
print(
    "Negative submission records:",
    len(negative_submission_dates)
)

print(
    "Negative date range:",
    negative_submission_dates["date_submitted"].min(),
    "to",
    negative_submission_dates["date_submitted"].max()
)

print(
    "Negative dates by banked status:"
)

print(
    negative_submission_dates["is_banked"]
    .value_counts(dropna=False)
)

Negative submission records: 2057
Negative date range: -11 to -1
Negative dates by banked status:
is_banked
1    1909
0     148
Name: count, dtype: int64


In [162]:
print(
    "Total records:",
    len(student_assessment)
)

print(
    "Records with scores:",
    student_assessment["score"].notna().sum()
)

print(
    "Records without scores:",
    student_assessment["score"].isna().sum()
)

print(
    "Percentage with scores:",
    round(
        student_assessment["score"].notna().mean() * 100,
        2
    )
)

print(
    "Percentage without scores:",
    round(
        student_assessment["score"].isna().mean() * 100,
        2
    )
)

Total records: 173912
Records with scores: 173739
Records without scores: 173
Percentage with scores: 99.9
Percentage without scores: 0.1


In [163]:
missing_score_assessments = (
    missing_score_details
    .groupby(
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "assessment_type",
            "date",
            "weight"
        ]
    )
    .size()
    .reset_index(name="missing_score_count")
    .sort_values(
        "missing_score_count",
        ascending=False
    )
)

missing_score_assessments

,id_assessment,code_module,code_presentation,assessment_type,date,weight,missing_score_count
34,25339,DDD,2013B,TMA,200.0,15.0,7
18,15013,BBB,2014B,TMA,194.0,18.0,7
56,34860,FFF,2013B,TMA,19.0,12.5,6
45,25363,DDD,2014J,TMA,41.0,10.0,5
3,14984,BBB,2013B,TMA,19.0,5.0,5
...,...,...,...,...,...,...,...
67,34887,FFF,2014B,TMA,52.0,12.5,1
66,34886,FFF,2014B,TMA,24.0,12.5,1
74,34903,FFF,2014J,TMA,199.0,25.0,1
75,37415,GGG,2013J,TMA,61.0,0.0,1


In [164]:
print(
    "Missing-score rows with invalid student IDs:",
    (
        ~missing_score_rows["id_student"].isin(
            student_info["id_student"]
        )
    ).sum()
)

Missing-score rows with invalid student IDs: 0


In [165]:
print(
    "Missing-score rows with invalid assessment IDs:",
    (
        ~missing_score_rows["id_assessment"].isin(
            assessments_clean["id_assessment"]
        )
    ).sum()
)

Missing-score rows with invalid assessment IDs: 0


In [166]:
student_assessment_clean = student_assessment.copy()

In [167]:
student_assessment_clean = (
    student_assessment_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

In [168]:
print(
    "Original rows:",
    len(student_assessment)
)

print(
    "Cleaned rows:",
    len(student_assessment_clean)
)

print(
    "Rows removed:",
    len(student_assessment) - len(student_assessment_clean)
)

Original rows: 173912
Cleaned rows: 173912
Rows removed: 0


In [169]:
print("Final rows:", len(student_assessment_clean))
print("Final columns:", len(student_assessment_clean.columns))

print(
    "Exact duplicates:",
    student_assessment_clean.duplicated().sum()
)

print(
    "Duplicate student-assessment pairs:",
    student_assessment_clean.duplicated(
        subset=[
            "id_assessment",
            "id_student"
        ]
    ).sum()
)

print("\nMissing values:")
print(student_assessment_clean.isnull().sum())

Final rows: 173912
Final columns: 5
Exact duplicates: 0
Duplicate student-assessment pairs: 0

Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64


In [170]:
print("Scores below 0:",
      (student_assessment_clean["score"] < 0).sum())

print("Scores above 100:",
      (student_assessment_clean["score"] > 100).sum())

Scores below 0: 0
Scores above 100: 0


In [171]:
print(
    "is_banked values:"
)
print(
    student_assessment_clean["is_banked"]
    .value_counts(dropna=False)
)

is_banked values:
is_banked
0    172003
1      1909
Name: count, dtype: int64


In [172]:
print(
    "Invalid assessment IDs:",
    (
        ~student_assessment_clean["id_assessment"].isin(
            assessments_clean["id_assessment"]
        )
    ).sum()
)

Invalid assessment IDs: 0


In [173]:
print("=== STUDENT ASSESSMENT CLEANING SUMMARY ===")

print("Original rows:", len(student_assessment))
print("Final rows:", len(student_assessment_clean))
print("Columns:", len(student_assessment_clean.columns))

print(
    "Exact duplicates:",
    student_assessment_clean.duplicated().sum()
)

print(
    "Duplicate student-assessment pairs:",
    student_assessment_clean.duplicated(
        subset=["id_assessment", "id_student"]
    ).sum()
)

print(
    "Missing scores:",
    student_assessment_clean["score"].isna().sum()
)

print(
    "Invalid assessment IDs:",
    (
        ~student_assessment_clean["id_assessment"].isin(
            assessments_clean["id_assessment"]
        )
    ).sum()
)

=== STUDENT ASSESSMENT CLEANING SUMMARY ===
Original rows: 173912
Final rows: 173912
Columns: 5
Exact duplicates: 0
Duplicate student-assessment pairs: 0
Missing scores: 173
Invalid assessment IDs: 0


In [174]:
student_assessment_clean["score"].describe()

count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64

In [175]:
print(
    student_assessment_clean[
        student_assessment_clean["score"].isna()
    ]["is_banked"].value_counts(dropna=False)
)

is_banked
0    172
1      1
Name: count, dtype: int64


In [176]:
assessment_student_ids = set(
    student_assessment_clean["id_student"]
)

info_student_ids = set(
    student_info["id_student"]
)

print(
    "Invalid student IDs:",
    len(assessment_student_ids - info_student_ids)
)

Invalid student IDs: 0


In [177]:
student_assessment_final = student_assessment_clean.copy()

print("Final student assessment rows:",
      len(student_assessment_final))

print("Final missing values:")
print(student_assessment_final.isnull().sum())

print("Final exact duplicates:",
      student_assessment_final.duplicated().sum())

Final student assessment rows: 173912
Final missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64
Final exact duplicates: 0


In [178]:
print("Final rows:", len(student_assessment_final))
print("Final columns:", len(student_assessment_final.columns))
print("Exact duplicates:", student_assessment_final.duplicated().sum())
print("Missing scores:", student_assessment_final["score"].isna().sum())

Final rows: 173912
Final columns: 5


Exact duplicates: 0
Missing scores: 173


In [179]:
print(
    student_assessment_final[
        student_assessment_final["score"].isna()
    ].head(10)
)

      id_assessment  id_student  date_submitted  is_banked  score
215            1752      721259              22          0    NaN
937            1754      260355             127          0    NaN
2364           1760     2606802             180          0    NaN
3358          14984      186780              77          0    NaN
3914          14984      531205              26          0    NaN
3943          14984      534151               7          0    NaN
4206          14984      549713              84          0    NaN
4311          14984      554393              55          0    NaN
4753          14985      186780              77          0    NaN
5722          14986       33666             117          0    NaN


In [180]:
print(
    "Score mean:",
    student_assessment_final["score"].mean()
)

print(
    "Score median:",
    student_assessment_final["score"].median()
)

Score mean: 75.7995729226023
Score median: 80.0


In [181]:
print("Student Assessment dataset is cleaned and validated.")

Student Assessment dataset is cleaned and validated.


In [182]:
assessment_students = set(
    student_assessment_final["id_student"]
)

info_students = set(
    student_info["id_student"]
)

print(
    "Assessment students missing from student info:",
    len(assessment_students - info_students)
)

Assessment students missing from student info: 0


In [183]:
registration_students = set(
    student_registration_clean["id_student"]
)

print(
    "Assessment students missing from registration:",
    len(assessment_students - registration_students)
)

Assessment students missing from registration: 0


In [184]:
assessment_ids = set(
    student_assessment_final["id_assessment"]
)

metadata_assessment_ids = set(
    assessments_clean["id_assessment"]
)

print(
    "Assessment IDs missing from metadata:",
    len(assessment_ids - metadata_assessment_ids)
)

Assessment IDs missing from metadata: 0


In [185]:
registration_students = set(
    student_registration_clean["id_student"]
)

print(
    "Registration students missing from student info:",
    len(registration_students - info_students)
)

Registration students missing from student info: 0


In [186]:
print("Students in student info:", len(info_students))
print("Students in registration:", len(registration_students))
print("Students in assessment:", len(assessment_students))

Students in student info: 28785
Students in registration: 28785
Students in assessment: 23369


In [187]:
print(
    "Student info students without assessments:",
    len(info_students - assessment_students)
)

Student info students without assessments: 5416


In [188]:
print(
    "Student info students without registration:",
    len(info_students - registration_students)
)


Student info students without registration: 0


In [189]:
print("=== CROSS-DATASET RELATIONSHIP VALIDATION ===")

print(
    "Assessment → Student Info:",
    len(assessment_students - info_students)
)

print(
    "Assessment → Registration:",
    len(assessment_students - registration_students)
)

print(
    "Assessment → Metadata:",
    len(assessment_ids - metadata_assessment_ids)
)

print(
    "Registration → Student Info:",
    len(registration_students - info_students)
)

=== CROSS-DATASET RELATIONSHIP VALIDATION ===
Assessment → Student Info: 0
Assessment → Registration: 0
Assessment → Metadata: 0
Registration → Student Info: 0


In [190]:
print(
    "Assessment coverage (%):",
    round(
        len(assessment_students) / len(info_students) * 100,
        2
    )
)

Assessment coverage (%): 81.18


In [191]:
print(
    "Students without assessment (%):",
    round(
        len(info_students - assessment_students)
        / len(info_students) * 100,
        2
    )
)

Students without assessment (%): 18.82


In [192]:
print(
    "Registration coverage (%):",
    round(
        len(registration_students) / len(info_students) * 100,
        2
    )
)

Registration coverage (%): 100.0


In [193]:
print("=== FINAL RELATIONSHIP SUMMARY ===")

print("Student Info students:", len(info_students))
print("Registration students:", len(registration_students))
print("Assessment students:", len(assessment_students))

print(
    "Students without assessments:",
    len(info_students - assessment_students)
)

print(
    "Students without registration:",
    len(info_students - registration_students)
)

print(
    "Invalid assessment → student relationships:",
    len(assessment_students - info_students)
)

print(
    "Invalid assessment → metadata relationships:",
    len(assessment_ids - metadata_assessment_ids)
)

=== FINAL RELATIONSHIP SUMMARY ===
Student Info students: 28785
Registration students: 28785
Assessment students: 23369
Students without assessments: 5416
Students without registration: 0
Invalid assessment → student relationships: 0
Invalid assessment → metadata relationships: 0


In [194]:
print(
    "Student Info modules:",
    sorted(student_info["code_module"].unique())
)

print(
    "Assessment modules:",
    sorted(assessments_clean["code_module"].unique())
)

print(
    "Student Assessment modules:",
    sorted(
        student_assessment_final
        .merge(
            assessments_clean[["id_assessment", "code_module"]],
            on="id_assessment",
            how="left"
        )["code_module"].dropna().unique()
    )
)

Student Info modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
Assessment modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
Student Assessment modules: ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']


In [195]:
assessment_with_metadata = student_assessment_final.merge(
    assessments_clean[
        ["id_assessment", "code_module", "code_presentation"]
    ],
    on="id_assessment",
    how="left"
)

print(
    "Invalid module/presentation relationships:",
    (
        ~assessment_with_metadata.set_index(
            ["code_module", "code_presentation"]
        ).index.isin(
            assessments_clean.set_index(
                ["code_module", "code_presentation"]
            ).index
        )
    ).sum()
)

Invalid module/presentation relationships: 0


In [196]:
assessment_with_metadata["code_module"].value_counts().sort_index()

code_module
AAA     3149
BBB    43032
CCC    18940
DDD    30864
EEE     7893
FFF    54815
GGG    15219
Name: count, dtype: int64

In [197]:
assessment_with_metadata["code_presentation"].value_counts().sort_index()

code_presentation
2013B    33824
2013J    49018
2014B    38896
2014J    52174
Name: count, dtype: int64

In [198]:
print(
    "Assessment records without metadata:",
    assessment_with_metadata["code_module"].isna().sum()
)

Assessment records without metadata: 0


In [199]:
print(
    "Assessment ID duplicates after merge:",
    assessment_with_metadata["id_assessment"].duplicated().sum()
)

Assessment ID duplicates after merge: 173724


In [200]:
print(
    "Assessment types:",
    sorted(assessments_clean["assessment_type"].unique())
)

Assessment types: ['CMA', 'Exam', 'TMA']


In [201]:
print("=== STEP 19 FINAL VALIDATION ===")

print("Student assessment rows:", len(student_assessment_final))
print("Metadata assessment rows:", len(assessments_clean))

print(
    "Records without metadata:",
    assessment_with_metadata["code_module"].isna().sum()
)

print(
    "Invalid assessment IDs:",
    len(assessment_ids - metadata_assessment_ids)
)

print(
    "Exact duplicates:",
    student_assessment_final.duplicated().sum()
)

=== STEP 19 FINAL VALIDATION ===
Student assessment rows: 173912
Metadata assessment rows: 206
Records without metadata: 0
Invalid assessment IDs: 0
Exact duplicates: 0


In [202]:
assessments_final = assessments_clean.copy()

In [203]:
vle_final = vle_clean.copy()

In [204]:
print("Student Info:", len(student_info))
print("Registration:", len(student_registration_clean))
print("VLE:", len(student_vle_exact_clean))
print("VLE Metadata:", len(vle_final))
print("Assessments:", len(assessments_final))
print("Student Assessment:", len(student_assessment_final))

Student Info: 32593
Registration: 32593
VLE: 9868110
VLE Metadata: 6364
Assessments: 206
Student Assessment: 173912


In [205]:
print("=== FINAL DATASET INTEGRITY CHECK ===")

print("Student Info missing:", student_info.isnull().sum().sum())
print("Registration missing:", student_registration_clean.isnull().sum().sum())
print("VLE missing:", student_vle_exact_clean.isnull().sum().sum())
print("VLE Metadata missing:", vle_final.isnull().sum().sum())
print("Assessments missing:", assessments_final.isnull().sum().sum())
print("Student Assessment missing:", student_assessment_final.isnull().sum().sum())

=== FINAL DATASET INTEGRITY CHECK ===
Student Info missing: 0
Registration missing: 22566
VLE missing: 0
VLE Metadata missing: 0
Assessments missing: 11
Student Assessment missing: 173


In [206]:
print("Registration missing date_unregistration:",
      student_registration_clean["date_unregistration"].isna().sum())

Registration missing date_unregistration: 22521


In [207]:
print("VLE metadata missing week_from:",
      vle_final["week_from"].isna().sum())

print("VLE metadata missing week_to:",
      vle_final["week_to"].isna().sum())

VLE metadata missing week_from: 0
VLE metadata missing week_to: 0


In [208]:
print("Assessment missing dates:",
      assessments_final["date"].isna().sum())

Assessment missing dates: 11


In [209]:
print("Student Assessment missing scores:",
      student_assessment_final["score"].isna().sum())

Student Assessment missing scores: 173


In [210]:
print("=== REGISTRATION FINAL VALIDATION ===")

print("Rows:", len(student_registration_clean))
print("Columns:", len(student_registration_clean.columns))
print("Exact duplicates:",
      student_registration_clean.duplicated().sum())
print("Missing values:")
print(student_registration_clean.isnull().sum())

=== REGISTRATION FINAL VALIDATION ===
Rows: 32593
Columns: 5
Exact duplicates: 0
Missing values:
code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64


In [211]:
print("=== VLE METADATA FINAL VALIDATION ===")

print("Rows:", len(vle_final))
print("Columns:", len(vle_final.columns))
print("Exact duplicates:", vle_final.duplicated().sum())

print("Missing values:")
print(vle_final.isnull().sum())

=== VLE METADATA FINAL VALIDATION ===
Rows: 6364
Columns: 6
Exact duplicates: 0
Missing values:
id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64


In [212]:
print("=== ASSESSMENTS FINAL VALIDATION ===")

print("Rows:", len(assessments_final))
print("Columns:", len(assessments_final.columns))
print("Exact duplicates:", assessments_final.duplicated().sum())

print("Missing values:")
print(assessments_final.isnull().sum())

=== ASSESSMENTS FINAL VALIDATION ===
Rows: 206
Columns: 6
Exact duplicates: 0
Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64


In [213]:
print("=== STUDENT ASSESSMENT FINAL VALIDATION ===")

print("Rows:", len(student_assessment_final))
print("Columns:", len(student_assessment_final.columns))
print("Exact duplicates:", student_assessment_final.duplicated().sum())

print("Missing values:")
print(student_assessment_final.isnull().sum())

=== STUDENT ASSESSMENT FINAL VALIDATION ===
Rows: 173912
Columns: 5


Exact duplicates: 0
Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64


In [214]:
print("=== REGISTRATION DATE VALIDATION ===")

print("Missing date_registration:",
      student_registration_clean["date_registration"].isnull().sum())

print("Missing date_unregistration:",
      student_registration_clean["date_unregistration"].isnull().sum())

print("Negative date_registration:",
      (student_registration_clean["date_registration"] < 0).sum())

print("Negative date_unregistration:",
      (student_registration_clean["date_unregistration"] < 0).sum())

=== REGISTRATION DATE VALIDATION ===
Missing date_registration: 45
Missing date_unregistration: 22521
Negative date_registration: 32312
Negative date_unregistration: 2678


In [215]:
print("=== VLE WEEK VALIDATION ===")

print("Missing week_from:",
      vle_final["week_from"].isnull().sum())

print("Missing week_to:",
      vle_final["week_to"].isnull().sum())

print("Rows with both weeks missing:",
      (
          vle_final["week_from"].isnull()
          & vle_final["week_to"].isnull()
      ).sum())

print("Rows with invalid week ranges:",
      (
          vle_final["week_from"].notna()
          & vle_final["week_to"].notna()
          & (vle_final["week_to"] < vle_final["week_from"])
      ).sum())

=== VLE WEEK VALIDATION ===
Missing week_from: 0
Missing week_to: 0
Rows with both weeks missing: 0
Rows with invalid week ranges: 0


In [216]:
print("=== ASSESSMENT DATE AND WEIGHT VALIDATION ===")

print("Missing assessment dates:",
      assessments_final["date"].isnull().sum())

print("Negative assessment dates:",
      (assessments_final["date"] < 0).sum())

print("Weights <= 0:",
      (assessments_final["weight"] <= 0).sum())

print("Weights > 100:",
      (assessments_final["weight"] > 100).sum())

=== ASSESSMENT DATE AND WEIGHT VALIDATION ===
Missing assessment dates: 11
Negative assessment dates: 0
Weights <= 0: 56
Weights > 100: 0


In [217]:
print("=== STUDENT ASSESSMENT QUALITY CHECK ===")

print("Missing scores:",
      student_assessment_final["score"].isnull().sum())

print("Scores below 0:",
      (student_assessment_final["score"] < 0).sum())

print("Scores above 100:",
      (student_assessment_final["score"] > 100).sum())

print("Invalid is_banked values:",
      (~student_assessment_final["is_banked"].isin([0, 1])).sum())

print("Invalid student IDs:",
      (~student_assessment_final["id_student"].isin(
          student_info["id_student"]
      )).sum())

=== STUDENT ASSESSMENT QUALITY CHECK ===
Missing scores: 173
Scores below 0: 0
Scores above 100: 0
Invalid is_banked values: 0
Invalid student IDs: 0


In [218]:
print("=== REGISTRATION DATE RELATIONSHIP ===")

print("Rows with both dates available:",
      (
          student_registration_clean["date_registration"].notna()
          & student_registration_clean["date_unregistration"].notna()
      ).sum())

print("Unregistration before registration:",
      (
          student_registration_clean["date_registration"].notna()
          & student_registration_clean["date_unregistration"].notna()
          & (
              student_registration_clean["date_unregistration"]
              < student_registration_clean["date_registration"]
          )
      ).sum())

=== REGISTRATION DATE RELATIONSHIP ===
Rows with both dates available: 10033
Unregistration before registration: 0


In [219]:
assessment_weight_check = (
    assessments_final
    .groupby(["code_module", "code_presentation"])["weight"]
    .sum()
)

print(assessment_weight_check)

code_module  code_presentation
AAA          2013J                200.0
             2014J                200.0
BBB          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
CCC          2014B                300.0
             2014J                300.0
DDD          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
EEE          2013J                200.0
             2014B                200.0
             2014J                200.0
FFF          2013B                200.0
             2013J                200.0
             2014B                200.0
             2014J                200.0
GGG          2013J                100.0
             2014B                100.0
             2014J                100.0
Name: weight, dtype: float64


In [220]:
print("=== VLE ACTIVITY TYPES ===")

print(
    vle_final["activity_type"]
    .value_counts()
    .sort_index()
)

=== VLE ACTIVITY TYPES ===
activity_type
dataplus            28
dualpane            20
externalquiz        26
folder               2
forumng            194
glossary            21
homepage            22
htmlactivity         4
oucollaborate       82
oucontent          996
ouelluminate        21
ouwiki              49
page               102
questionnaire       61
quiz               127
repeatactivity       5
resource          2660
sharedsubpage        3
subpage           1055
url                886
Name: count, dtype: int64


In [221]:
print("=== STUDENT ASSESSMENT SCORE DISTRIBUTION ===")

print(
    student_assessment_final["score"]
    .describe()
)

print(
    "Scores exactly 0:",
    (student_assessment_final["score"] == 0).sum()
)

print(
    "Scores exactly 100:",
    (student_assessment_final["score"] == 100).sum()
)

=== STUDENT ASSESSMENT SCORE DISTRIBUTION ===
count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64
Scores exactly 0: 329
Scores exactly 100: 18813


In [222]:
print("=== REGISTRATION DATE RANGE ===")

print(
    student_registration_clean[
        ["date_registration", "date_unregistration"]
    ].describe()
)

print(
    "Registrations before module start:",
    (student_registration_clean["date_registration"] < 0).sum()
)

print(
    "Unregistrations before module start:",
    (student_registration_clean["date_unregistration"] < 0).sum()
)

=== REGISTRATION DATE RANGE ===
       date_registration  date_unregistration
count       32548.000000         10072.000000
mean          -69.411300            49.757645
std            49.260522            82.460890
min          -322.000000          -365.000000
25%          -100.000000            -2.000000
50%           -57.000000            27.000000
75%           -29.000000           109.000000
max           167.000000           444.000000
Registrations before module start: 32312
Unregistrations before module start: 2678


In [223]:
print("=== ASSESSMENT WEIGHT SUMMARY ===")

assessment_weight_summary = (
    assessments_final
    .groupby("code_module")["weight"]
    .agg(["count", "sum", "min", "max"])
)

print(assessment_weight_summary)

=== ASSESSMENT WEIGHT SUMMARY ===
             count    sum   min    max
code_module                           
AAA             12  400.0  10.0  100.0
BBB             42  800.0   0.0  100.0
CCC             20  600.0   2.0  100.0
DDD             35  800.0   2.0  100.0
EEE             15  600.0  16.0  100.0
FFF             52  800.0   0.0  100.0
GGG             30  300.0   0.0  100.0


In [224]:
print("=== VLE MISSING WEEKS BY ACTIVITY ===")

vle_missing_weeks = (
    vle_final[
        vle_final["week_from"].isnull()
        & vle_final["week_to"].isnull()
    ]
    .groupby("activity_type")
    .size()
    .sort_values(ascending=False)
)

print(vle_missing_weeks)

=== VLE MISSING WEEKS BY ACTIVITY ===
Series([], dtype: int64)


In [225]:
print("=== FINAL DATASET SUMMARY ===")

datasets_summary = {
    "Student Info": len(student_info),
    "Registration": len(student_registration_clean),
    "VLE Activity": len(student_vle_exact_clean),
    "VLE Metadata": len(vle_final),
    "Assessments": len(assessments_final),
    "Student Assessment": len(student_assessment_final)
}

for name, rows in datasets_summary.items():
    print(f"{name}: {rows:,} rows")

print()
print("Unique students:", student_info["id_student"].nunique())
print("Unique assessments:", assessments_final["id_assessment"].nunique())
print("Unique VLE sites:", vle_final["id_site"].nunique())

=== FINAL DATASET SUMMARY ===
Student Info: 32,593 rows
Registration: 32,593 rows
VLE Activity: 9,868,110 rows
VLE Metadata: 6,364 rows
Assessments: 206 rows
Student Assessment: 173,912 rows

Unique students: 28785
Unique assessments: 206
Unique VLE sites: 6364


In [226]:
registration_final = student_registration_clean.copy()

print("=== REGISTRATION FINAL ===")
print("Rows:", len(registration_final))
print("Columns:", len(registration_final.columns))
print("Exact duplicates:", registration_final.duplicated().sum())
print("Missing values:")
print(registration_final.isnull().sum())

=== REGISTRATION FINAL ===
Rows: 32593
Columns: 5
Exact duplicates: 0
Missing values:
code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64


In [227]:
print("=== VLE ACTIVITY FINAL ===")

print("Rows:", len(student_vle_exact_clean))
print("Columns:", len(student_vle_exact_clean.columns))
print("Exact duplicates:", student_vle_exact_clean.duplicated().sum())

print("Unique students:",
      student_vle_exact_clean["id_student"].nunique())

print("Unique sites:",
      student_vle_exact_clean["id_site"].nunique())

print("Missing values:")
print(student_vle_exact_clean.isnull().sum())

=== VLE ACTIVITY FINAL ===
Rows: 9868110
Columns: 6
Exact duplicates: 0
Unique students: 26074
Unique sites: 6268
Missing values:
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64


In [228]:
print("=== ASSESSMENTS FINAL ===")

print("Rows:", len(assessments_final))
print("Columns:", len(assessments_final.columns))
print("Exact duplicates:", assessments_final.duplicated().sum())

print("Unique assessment IDs:",
      assessments_final["id_assessment"].nunique())

print("Missing dates:",
      assessments_final["date"].isnull().sum())

print("Missing values:")
print(assessments_final.isnull().sum())

=== ASSESSMENTS FINAL ===
Rows: 206
Columns: 6
Exact duplicates: 0
Unique assessment IDs: 206
Missing dates: 11
Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64


In [229]:
final_datasets = {
    "student_info": student_info.copy(),
    "registration": registration_final.copy(),
    "student_vle": student_vle_exact_clean.copy(),
    "vle": vle_final.copy(),
    "assessments": assessments_final.copy(),
    "student_assessment": student_assessment_final.copy()
}

print("=== FINAL DATASETS READY ===")

for name, df in final_datasets.items():
    print(
        f"{name}: {df.shape[0]:,} rows × "
        f"{df.shape[1]} columns"
    )

=== FINAL DATASETS READY ===
student_info: 32,593 rows × 12 columns
registration: 32,593 rows × 5 columns
student_vle: 9,868,110 rows × 6 columns
vle: 6,364 rows × 6 columns
assessments: 206 rows × 6 columns
student_assessment: 173,912 rows × 5 columns


# 7. Export Cleaned Datasets

The validated datasets are exported to the `data/processed/` directory.

The raw datasets in `data/raw/` remain unchanged.

These processed datasets will be used for Exploratory Data Analysis, feature engineering, machine learning, and Power BI.

In [230]:
import os

processed_path = "../data/processed"

os.makedirs(processed_path, exist_ok=True)

print("Processed folder:", os.path.abspath(processed_path))

Processed folder: c:\Users\Rushi\OneDrive\Desktop\Major Projects\EdTech-Learning-Analytics\data\processed


In [231]:
student_info.to_csv(
    os.path.join(processed_path, "student_info_cleaned.csv"),
    index=False
)

In [232]:
registration_final.to_csv(
    os.path.join(processed_path, "student_registration_cleaned.csv"),
    index=False
)

In [233]:
student_vle_exact_clean.to_csv(
    os.path.join(processed_path, "student_vle_cleaned.csv"),
    index=False
)

In [234]:
vle_final.to_csv(
    os.path.join(processed_path, "vle_cleaned.csv"),
    index=False
)

In [235]:
assessments_final.to_csv(
    os.path.join(processed_path, "assessments_cleaned.csv"),
    index=False
)

In [236]:
student_assessment_final.to_csv(
    os.path.join(processed_path, "student_assessment_cleaned.csv"),
    index=False
)

In [237]:
print("=== PROCESSED FILES ===")

for file in sorted(os.listdir(processed_path)):
    print(file)

=== PROCESSED FILES ===
assessments_cleaned.csv
student_assessment_cleaned.csv
student_info_cleaned.csv
student_registration_cleaned.csv
student_vle_cleaned.csv
vle_cleaned.csv


In [238]:
print("=== SAVED DATASET VALIDATION ===")

for file in sorted(os.listdir(processed_path)):
    if file.endswith(".csv"):
        df_check = pd.read_csv(
            os.path.join(processed_path, file)
        )
        print(f"{file}: {df_check.shape[0]:,} rows × {df_check.shape[1]} columns")

=== SAVED DATASET VALIDATION ===
assessments_cleaned.csv: 206 rows × 6 columns
student_assessment_cleaned.csv: 173,912 rows × 5 columns
student_info_cleaned.csv: 32,593 rows × 12 columns
student_registration_cleaned.csv: 32,593 rows × 5 columns
student_vle_cleaned.csv: 9,868,110 rows × 6 columns
vle_cleaned.csv: 6,364 rows × 6 columns
